# Reproduce Directional Analysis 2 / final Figure S4

This notebook provides the interactive SCIViewer workflow for Directional Analysis 2. In the local manuscript figure folder, the corresponding image file was named `png_files/Fig_S3.png`; in the final manuscript, this SCIViewer directional-analysis figure is Figure S4.

Use it in one of two ways:

1. **Interactive mode:** load the AnnData object, open SCIViewer, recreate the Directional Analysis 2 selection, export selected cells and projection-correlation results, then regenerate the tables and figure-panel outputs.
2. **Saved-export mode:** use the saved SCIViewer export from 12Jan25 to regenerate the submitted supplemental tables and figure-panel data.

Large `.h5ad` files and generated outputs should remain local or in GEO, not committed to GitHub.


In [ ]:
from pathlib import Path
import os
import sys

HERE = Path.cwd().resolve()
REPO_ROOT = HERE.parent if HERE.name == "notebooks" else HERE

# Set HCMV_DATA_DIR to the folder containing downloaded data/intermediate files.
DATA_DIR = Path(os.environ.get("HCMV_DATA_DIR", REPO_ROOT / "data")).expanduser().resolve()
SUPPLEMENT_DIR = Path(os.environ.get("HCMV_SUPPLEMENT_DIR", REPO_ROOT / "supplemental")).expanduser().resolve()

OUTPUT_DIR = REPO_ROOT / "outputs" / "directional_analysis_2"
TABLE_OUTPUT_DIR = REPO_ROOT / "outputs" / "tables"
FIGURE_OUTPUT_DIR = REPO_ROOT / "outputs" / "figures" / "directional_analysis_2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The saved 12Jan25 selected-cell indices map to this 389-cell 3 dpi infected subset.
SCIVIEWER_ADATA = DATA_DIR / "infected_cells_3dpi_cmv_data.h5ad"

# Alternate 389-cell subset with the same cell order but fewer genes.
ALT_SCIVIEWER_ADATA = DATA_DIR / "cmv.srt.soupx.filt.dpi3Infected.only_v3.h5ad"

# The manuscript dotplots use all 3 dpi infection-state / viral-percent bins.
DOTPLOT_ADATA = DATA_DIR / "cmv.srt.soupx.filt.h5ad"

# Saved SCIViewer Directional Analysis 2 exports.
SAVED_SELECTED_CELLS = DATA_DIR / "selected_cells_12Jan25_dir1_3dpiinfect_cells_hostonly.csv"
SAVED_PROJ_CORR_XLSX = DATA_DIR / "results_12Jan25_dir1_3dpiinf_cells_hostonly.proj_correlation.xlsx"
SAVED_PROJ_CORR_CSV = DATA_DIR / "results_12Jan25_dir1_3dpiinf_cells_hostonly.proj_correlation.csv"

# New local exports made by this notebook.
NEW_SELECTED_CELLS = OUTPUT_DIR / "selected_cells_directional_analysis_2_recreated.csv"
NEW_PROJ_CORR_CSV = OUTPUT_DIR / "directional_analysis_2_recreated.proj_correlation.csv"

print("Repo root:", REPO_ROOT)
print("Data dir:", DATA_DIR)
print("SCIViewer AnnData exists:", SCIVIEWER_ADATA.exists())
print("Saved selected-cell file exists:", SAVED_SELECTED_CELLS.exists())
print("Saved projection-correlation workbook exists:", SAVED_PROJ_CORR_XLSX.exists())


## Load the 3 dpi infected-cell AnnData object for the interactive SCIViewer step

Directional Analysis 2 was performed on the 3 dpi infected-cell subset, not on all cells. The saved 12Jan25 selected-cell export has indices that map exactly to `infected_cells_3dpi_cmv_data.h5ad` and `cmv.srt.soupx.filt.dpi3Infected.only_v3.h5ad`, but not to the all-cell `cmv.srt.soupx.filt.updated.h5ad` object.

This notebook therefore opens `infected_cells_3dpi_cmv_data.h5ad` for SCIViewer. The alternate `_dpi3Infected.only_v3.h5ad` subset has the same cell order but fewer genes, so the 20,388-gene `infected_cells_3dpi_cmv_data.h5ad` file is the safer default.


In [ ]:
import pandas as pd

try:
    import scanpy as sc
except ImportError as exc:
    raise ImportError("Install scanpy/anndata in this Jupyter environment before running the interactive workflow.") from exc

adata = sc.read_h5ad(SCIVIEWER_ADATA)
print("Loaded:", SCIVIEWER_ADATA.name)
print("Shape:", adata.shape)

for column in ["dpi", "virus.presence", "Infection_state", "Infection_state_bkgd"]:
    if column in adata.obs:
        print(column, adata.obs[column].astype(str).value_counts(dropna=False).to_dict())

if SAVED_SELECTED_CELLS.exists():
    saved_selected = pd.read_csv(SAVED_SELECTED_CELLS)
    checked = saved_selected.head(20).apply(
        lambda row: int(row["index"]) < adata.n_obs and adata.obs_names[int(row["index"])] == row["cell_name"],
        axis=1,
    )
    print("First 20 saved selected-cell indices match this object:", bool(checked.all()))

adata


## Open SCIViewer and recreate Directional Analysis 2

Run the next cell to open SCIViewer on the 389-cell 3 dpi infected-cell subset. Recreate the Directional Analysis 2 selection shown in final Figure S4B. When you are satisfied with the direction/selected cells, continue to the export cell below.

The saved selection used for the manuscript is `selected_cells_12Jan25_dir1_3dpiinfect_cells_hostonly.csv`. You can compare against it after exporting.


In [ ]:
import os

if 'JAVA_HOME' not in os.environ:
    conda_java_home = Path(os.environ.get('CONDA_PREFIX', '')) / 'lib' / 'jvm'
    if conda_java_home.exists():
        os.environ['JAVA_HOME'] = str(conda_java_home)
%gui osx
%load_ext py5

try:
    from sciviewer import SCIViewer
except ImportError as exc:
    raise ImportError(
        "SCIViewer is not installed in this Jupyter kernel. "
        "Select the `Python (HCMV sciviewer)` kernel, restart, and run from the top."
    ) from exc

svobj_dir2 = SCIViewer(adata, embedding_name="X_umap", use_raw=False)
svobj_dir2.explore_data()


## Export the recreated SCIViewer selection

Run this after you have recreated the directional selection in SCIViewer. It writes local files under `outputs/directional_analysis_2/`, which Git ignores.


In [ ]:
selected = pd.DataFrame(svobj_dir2.selected_cells)
selected.to_csv(NEW_SELECTED_CELLS, index=False)

proj_corr = svobj_dir2.results_proj_correlation.dropna(subset=["P"]).copy()
proj_corr.to_csv(NEW_PROJ_CORR_CSV, index=True)

print("Wrote:", NEW_SELECTED_CELLS)
print("Wrote:", NEW_PROJ_CORR_CSV)
print("Selected cells:", selected.shape)
print("Projection-correlation rows:", proj_corr.shape)
proj_corr.sort_values("P").head()


## Choose source for downstream reproduction

Use `USE_SAVED_EXPORT = True` to regenerate the submitted tables and figure-panel data from the saved intermediate workbook. Set it to `False` after exporting a newly recreated SCIViewer selection above.


In [ ]:
USE_SAVED_EXPORT = True

projection_correlation_source = SAVED_PROJ_CORR_XLSX if USE_SAVED_EXPORT else NEW_PROJ_CORR_CSV
print("Using projection-correlation source:", projection_correlation_source)
print("Exists:", projection_correlation_source.exists())


## Regenerate Table S6 and Table S7

This imports the cleaned deterministic helper functions from `scripts/directional_analysis/reproduce_directional_analysis_2.py`. Table S6 and Table S7 are the manuscript supplemental outputs for Directional Analysis 2.


In [ ]:
sys.path.insert(0, str(REPO_ROOT / "scripts" / "directional_analysis"))
from reproduce_directional_analysis_2 import write_table_s6, write_table_s7

TABLE_S6_OUT = TABLE_OUTPUT_DIR / "Table_S6_supplement_directional_analysis_2_raw_results.xlsx"
TABLE_S7_OUT = TABLE_OUTPUT_DIR / "Table_S7_supplement_GSEA_directional_analysis_2.xlsx"

write_table_s6(projection_correlation_source, TABLE_S6_OUT)
write_table_s7(SAVED_PROJ_CORR_XLSX, TABLE_S7_OUT)

print("Wrote:", TABLE_S6_OUT)
print("Wrote:", TABLE_S7_OUT)


## Verify against submitted supplemental tables, if available locally

This optional QA cell compares regenerated tables with local copies of the submitted supplemental tables, when those files are available.


In [ ]:
expected_s6 = SUPPLEMENT_DIR / "Table_S6_supplement_directional_analysis_2_raw_results.xlsx"
expected_s7 = SUPPLEMENT_DIR / "Table_S7_supplement_GSEA_directional_analysis_2.xlsx"

def compare_excel_values(expected_path, actual_path):
    if not expected_path.exists():
        print("Missing expected file, skipping:", expected_path)
        return
    expected = pd.read_excel(expected_path).fillna("").astype(str)
    actual = pd.read_excel(actual_path).fillna("").astype(str)
    common = [col for col in expected.columns if col in actual.columns]
    print(expected_path.name)
    print("  expected shape:", expected.shape)
    print("  actual shape:  ", actual.shape)
    print("  columns equal: ", list(expected.columns) == list(actual.columns))
    print("  values equal:  ", expected[common].equals(actual[common]))

compare_excel_values(expected_s6, TABLE_S6_OUT)
compare_excel_values(expected_s7, TABLE_S7_OUT)


## Plot Directional Analysis 2 GSEA panel

This creates the final Figure S4E-style GSEA NES bar plot from the regenerated Table S7 workbook. The final Figure S4 panel title says FDR <= 0.1, so this notebook uses `fdr_cutoff=0.1`.


In [ ]:
sys.path.insert(0, str(REPO_ROOT / "scripts" / "figures"))
from plot_directional_gsea_panel import load_gsea_table, select_pathways, plot_gsea_bar, write_selected_pathways

DA2_GSEA_PANEL_SVG = FIGURE_OUTPUT_DIR / "directional_analysis_2_gsea_fdr_0.1.svg"
DA2_GSEA_PANEL_PNG = FIGURE_OUTPUT_DIR / "directional_analysis_2_gsea_fdr_0.1.png"
DA2_GSEA_PANEL_TSV = FIGURE_OUTPUT_DIR / "directional_analysis_2_gsea_fdr_0.1_plotted_pathways.tsv"

gsea_table = load_gsea_table(TABLE_S7_OUT)
plotted_pathways = select_pathways(gsea_table, fdr_cutoff=0.1, top_n=30)
plot_gsea_bar(plotted_pathways, DA2_GSEA_PANEL_SVG, title="Directional Analysis 2 GSEA")
plot_gsea_bar(plotted_pathways, DA2_GSEA_PANEL_PNG, title="Directional Analysis 2 GSEA")
write_selected_pathways(plotted_pathways, DA2_GSEA_PANEL_TSV)

print("Wrote:", DA2_GSEA_PANEL_SVG)
print("Wrote:", DA2_GSEA_PANEL_PNG)
print("Wrote:", DA2_GSEA_PANEL_TSV)
print("Plotted pathways:", plotted_pathways.shape[0])
display(plotted_pathways[["Term", "NES", "FDR q-val"]])


## Plot Directional Analysis 2 top-gene dotplots

These reproduce the final Figure S4C/D-style dotplots for the positively and negatively correlated genes from Directional Analysis 2. The SCIViewer selection uses the 389-cell 3 dpi infected-cell subset above, but the dotplots are drawn across all 3 dpi infection-state groups from `cmv.srt.soupx.filt.h5ad`.

The grouping is: Mock, Bystander, Marginal Infection, then High Infection cells binned by percent HCMV transcripts. Dot color is z-scored average expression per gene, clipped to -2 to 2; dot size is percent of cells expressing the gene. Expression is read from `adata.raw` and summarized with Seurat DotPlot-style averaging to mirror the original R/scCustomize plotting convention.


In [ ]:
import importlib
import anndata as ad
import plot_directional_gene_dotplots as dotplots

dotplots = importlib.reload(dotplots)

dotplot_adata = ad.read_h5ad(DOTPLOT_ADATA)
dotplot_adata.obs["figure5_dotplot_group"] = dotplots.make_dotplot_group(dotplot_adata.obs, dpi="3dpi")
dotplot_adata = dotplot_adata[~pd.isna(dotplot_adata.obs["figure5_dotplot_group"]), :].copy()
dotplot_correlations = dotplots.load_correlation_table(TABLE_S6_OUT)
dotplot_results = []

for direction, top_n, title in [
    ("positive", 30, "sciViewer top positively correlated\ngene expression in dpi3 highly infected cells"),
    ("negative", 30, "sciViewer top negatively correlated\ngene expression in dpi3 highly infected cells"),
]:
    result = dotplots.build_dotplot(
        dotplot_adata,
        dotplot_correlations,
        direction=direction,
        output_dir=FIGURE_OUTPUT_DIR,
        top_n=top_n,
        use_raw=True,
        file_prefix="directional_analysis_2",
        title=title,
        figure_order=True,
        analysis="directional_analysis_2",
        average_method="seurat",
    )
    dotplot_results.append(result)
    print("Wrote:", result.figure_path)
    print("Wrote:", result.table_path)

[(r.direction, r.figure_path.name, r.table_path.name, len(r.genes)) for r in dotplot_results]


## Optional: compare a recreated selected-cell export to the original saved selection

Run this only after exporting a recreated selection. Exact equality is not required for a new interactive selection, but this comparison is useful for checking whether the same direction was recovered.


In [ ]:
if NEW_SELECTED_CELLS.exists() and SAVED_SELECTED_CELLS.exists():
    old = pd.read_csv(SAVED_SELECTED_CELLS).fillna("")
    new = pd.read_csv(NEW_SELECTED_CELLS).fillna("")
    print("Original selected cells shape:", old.shape)
    print("Recreated selected cells shape:", new.shape)
    common_cols = [col for col in old.columns if col in new.columns]
    if common_cols and old.shape == new.shape:
        print("Shared-column values equal:", old[common_cols].astype(str).equals(new[common_cols].astype(str)))
    display(old.head())
    display(new.head())
else:
    print("Selection comparison skipped. Export a recreated selection first, and confirm the saved selected-cell CSV exists.")
